# Semantic Canonicalization Slice

Purpose: prove the first semantic canonicalization replacement slice over promoted assertions, including explicit repair of bad promoted graph state and measurable before/after improvement.

Mode: proof

Related docs:
- `docs/plans/0001_successor_roadmap.md`
- `docs/plans/0008_phase13_semantic_canonicalization_shape.md`
- `docs/adr/0014-replace-the-v1-semantic-stack-with-pack-driven-canonicalization-and-explicit-recanonicalization.md`


## Phase 1

Purpose: seed one canonical DoDAF promoted assertion and mutate it into a legacy alias form.

Input -> output: `review_seed_inputs -> noncanonical_promoted_assertion`

Acceptance criteria:
1. A real promoted assertion exists before repair.
2. The persisted promoted graph state is explicitly mutated into a noncanonical predicate/role form.
3. The proof keeps the bad state inspectable instead of hiding it inside setup.

status: `proven`

execution_mode: `live`


In [1]:
from __future__ import annotations

import contextlib
import io
import json
import sqlite3
from pathlib import Path
from tempfile import TemporaryDirectory

from onto_canon6 import cli as cli_module
from onto_canon6.pipeline import ReviewService


In [2]:
workspace = TemporaryDirectory()
workspace_path = Path(workspace.name)
review_db_path = workspace_path / 'review.sqlite3'
overlay_root = workspace_path / 'ontology_overlays'

def run_cli(args: list[str]) -> tuple[int, str, str]:
    stdout_buffer = io.StringIO()
    stderr_buffer = io.StringIO()
    with contextlib.redirect_stdout(stdout_buffer), contextlib.redirect_stderr(stderr_buffer):
        exit_code = cli_module.main(args)
    return exit_code, stdout_buffer.getvalue(), stderr_buffer.getvalue()

review_service = ReviewService(
    db_path=review_db_path,
    overlay_root=overlay_root,
    default_acceptance_policy='record_only',
)
submission = review_service.submit_candidate_assertion(
    payload={
        'predicate': 'dodaf:operational_node_exchanges_information',
        'roles': {
            'source_node': [
                {'entity_id': 'ent:node:source', 'entity_type': 'dm2:OperationalNode'}
            ],
            'target_node': [
                {'entity_id': 'ent:node:target', 'entity_type': 'dm2:OperationalNode'}
            ],
            'information_element': [
                {'entity_id': 'ent:info:message', 'entity_type': 'dm2:InformationElement'}
            ],
        },
    },
    profile_id='dodaf_minimal_strict',
    profile_version='0.1.0',
    submitted_by='analyst:notebook',
    source_kind='text_file',
    source_ref='notes/dodaf_semantic_notebook.txt',
    source_text='Node A exchanges Message M with Node B.',
)
accepted = review_service.review_candidate(
    candidate_id=submission.candidate.candidate_id,
    decision='accepted',
    actor_id='analyst:reviewer',
)
promote_exit, promote_stdout, promote_stderr = run_cli([
    'promote-candidate',
    '--candidate-id', accepted.candidate_id,
    '--actor-id', 'analyst:graph-promoter',
    '--review-db-path', str(review_db_path),
    '--output', 'json',
])
assert promote_exit == 0, promote_stderr
promotion = json.loads(promote_stdout)
assertion_id = promotion['assertion']['assertion_id']

with sqlite3.connect(review_db_path) as conn:
    conn.execute(
        '''
        UPDATE promoted_graph_assertions
        SET predicate = ?,
            normalized_body_json = ?
        WHERE assertion_id = ?
        ''',
        (
            'OperationalNodeExchangesInformation',
            json.dumps(
                {
                    'predicate': 'OperationalNodeExchangesInformation',
                    'roles': {
                        'source': [
                            {
                                'kind': 'entity',
                                'entity_id': 'ent:node:source',
                                'entity_type': 'dm2:OperationalNode',
                            }
                        ],
                        'target': [
                            {
                                'kind': 'entity',
                                'entity_id': 'ent:node:target',
                                'entity_type': 'dm2:OperationalNode',
                            }
                        ],
                        'information': [
                            {
                                'kind': 'entity',
                                'entity_id': 'ent:info:message',
                                'entity_type': 'dm2:InformationElement',
                            }
                        ],
                    },
                },
                sort_keys=True,
            ),
            assertion_id,
        ),
    )
    conn.execute(
        '''
        UPDATE promoted_graph_role_fillers
        SET role_id = CASE role_id
            WHEN 'source_node' THEN 'source'
            WHEN 'target_node' THEN 'target'
            WHEN 'information_element' THEN 'information'
            ELSE role_id
        END
        WHERE assertion_id = ?
        ''',
        (assertion_id,),
    )

before_exit, before_stdout, before_stderr = run_cli([
    'list-promoted-assertions',
    '--review-db-path', str(review_db_path),
    '--output', 'json',
])
assert before_exit == 0, before_stderr
before_promoted_assertion = json.loads(before_stdout)[0]
noncanonical_promoted_assertion = {
    'assertion_id': assertion_id,
    'before_promoted_assertion': before_promoted_assertion,
    'before_exact_canonical_match': int(
        before_promoted_assertion['predicate'] == 'dodaf:operational_node_exchanges_information'
    ),
}
noncanonical_promoted_assertion


{'assertion_id': 'gassert_dc7c9bc024bf40c19f62559b',
 'before_promoted_assertion': {'assertion_id': 'gassert_dc7c9bc024bf40c19f62559b',
  'claim_text': None,
  'normalized_body': {'predicate': 'OperationalNodeExchangesInformation',
   'roles': {'information': [{'entity_id': 'ent:info:message',
      'entity_type': 'dm2:InformationElement',
      'kind': 'entity'}],
    'source': [{'entity_id': 'ent:node:source',
      'entity_type': 'dm2:OperationalNode',
      'kind': 'entity'}],
    'target': [{'entity_id': 'ent:node:target',
      'entity_type': 'dm2:OperationalNode',
      'kind': 'entity'}]}},
  'predicate': 'OperationalNodeExchangesInformation',
  'profile': {'profile_id': 'dodaf_minimal_strict',
   'profile_version': '0.1.0'},
  'promoted_at': '2026-03-18T20:09:00.832359+00:00',
  'promoted_by': 'analyst:graph-promoter',
  'source_candidate_id': 'cand_6e14a4f8760b4b43ada34327'},
 'before_exact_canonical_match': 0}

## Phase 2

Purpose: repair the promoted assertion through the explicit recanonicalization seam.

Input -> output: `noncanonical_promoted_assertion -> repaired_promoted_assertion`

Acceptance criteria:
1. The repair runs through the real CLI-backed semantic service.
2. Predicate and role ids are rewritten to the pack-canonical form.
3. A persisted recanonicalization event is created.

status: `proven`

execution_mode: `live`


In [3]:
repair_exit, repair_stdout, repair_stderr = run_cli([
    'recanonicalize-promoted-assertion',
    '--assertion-id', assertion_id,
    '--actor-id', 'analyst:semantic-repair',
    '--reason', 'Normalize legacy DoDAF alias ids.',
    '--review-db-path', str(review_db_path),
    '--output', 'json',
])
assert repair_exit == 0, repair_stderr
repaired_promoted_assertion = json.loads(repair_stdout)
assert repaired_promoted_assertion['status'] == 'rewritten'
assert repaired_promoted_assertion['assertion']['predicate'] == 'dodaf:operational_node_exchanges_information'
assert {filler['role_id'] for filler in repaired_promoted_assertion['role_fillers']} == {
    'source_node',
    'target_node',
    'information_element',
}
assert repaired_promoted_assertion['event']['before_predicate'] == 'OperationalNodeExchangesInformation'
repaired_promoted_assertion


{'assertion': {'assertion_id': 'gassert_dc7c9bc024bf40c19f62559b',
  'claim_text': None,
  'normalized_body': {'predicate': 'dodaf:operational_node_exchanges_information',
   'roles': {'information_element': [{'entity_id': 'ent:info:message',
      'entity_type': 'dm2:InformationElement',
      'kind': 'entity'}],
    'source_node': [{'entity_id': 'ent:node:source',
      'entity_type': 'dm2:OperationalNode',
      'kind': 'entity'}],
    'target_node': [{'entity_id': 'ent:node:target',
      'entity_type': 'dm2:OperationalNode',
      'kind': 'entity'}]}},
  'predicate': 'dodaf:operational_node_exchanges_information',
  'profile': {'profile_id': 'dodaf_minimal_strict',
   'profile_version': '0.1.0'},
  'promoted_at': '2026-03-18T20:09:00.832359+00:00',
  'promoted_by': 'analyst:graph-promoter',
  'source_candidate_id': 'cand_6e14a4f8760b4b43ada34327'},
 'event': {'actor_id': 'analyst:semantic-repair',
  'after_body': {'predicate': 'dodaf:operational_node_exchanges_information',
   'ro

## Phase 3

Purpose: export the semantic repair report and show measurable before/after improvement.

Input -> output: `repaired_promoted_assertion -> semantic_canonicalization_report`

Acceptance criteria:
1. The persisted event trail is inspectable.
2. The report exposes the repaired promoted assertion state.
3. The notebook emits a measurable before/after canonicalization summary.

status: `proven`

execution_mode: `live`


In [4]:
events_exit, events_stdout, events_stderr = run_cli([
    'list-recanonicalization-events',
    '--review-db-path', str(review_db_path),
    '--output', 'json',
])
assert events_exit == 0, events_stderr
recanonicalization_events = json.loads(events_stdout)

report_exit, report_stdout, report_stderr = run_cli([
    'export-semantic-canonicalization-report',
    '--review-db-path', str(review_db_path),
    '--output', 'json',
])
assert report_exit == 0, report_stderr
semantic_canonicalization_report = json.loads(report_stdout)
quality_summary = {
    'before_exact_canonical_match': noncanonical_promoted_assertion['before_exact_canonical_match'],
    'after_exact_canonical_match': int(
        semantic_canonicalization_report['assertion_bundles'][0]['assertion']['predicate']
        == 'dodaf:operational_node_exchanges_information'
    ),
    'event_count': semantic_canonicalization_report['summary']['total_recanonicalization_events'],
}
assert quality_summary['before_exact_canonical_match'] == 0
assert quality_summary['after_exact_canonical_match'] == 1
assert quality_summary['event_count'] == 1
{
    'events': recanonicalization_events,
    'report': semantic_canonicalization_report,
    'quality_summary': quality_summary,
}


{'events': [{'actor_id': 'analyst:semantic-repair',
   'after_body': {'predicate': 'dodaf:operational_node_exchanges_information',
    'roles': {'information_element': [{'entity_id': 'ent:info:message',
       'entity_type': 'dm2:InformationElement',
       'kind': 'entity'}],
     'source_node': [{'entity_id': 'ent:node:source',
       'entity_type': 'dm2:OperationalNode',
       'kind': 'entity'}],
     'target_node': [{'entity_id': 'ent:node:target',
       'entity_type': 'dm2:OperationalNode',
       'kind': 'entity'}]}},
   'after_predicate': 'dodaf:operational_node_exchanges_information',
   'assertion_id': 'gassert_dc7c9bc024bf40c19f62559b',
   'before_body': {'predicate': 'OperationalNodeExchangesInformation',
    'roles': {'information': [{'entity_id': 'ent:info:message',
       'entity_type': 'dm2:InformationElement',
       'kind': 'entity'}],
     'source': [{'entity_id': 'ent:node:source',
       'entity_type': 'dm2:OperationalNode',
       'kind': 'entity'}],
     'target

In [5]:
workspace.cleanup()
'cleanup_complete'

'cleanup_complete'